# 05 | Juntar as quatro fontes

Agora que 01, 02, 03 e 04 salvaram seus dados, podemos combinar tudo. **Não é necessário sair deste notebook e voltar depois.**

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Carregar tabelas

O campo `state` diz o estado e `year` o ano. Eles são usados como chaves para juntar as bases.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
p = ROOT / "data"
cdc = pd.read_csv(p/"raw/cdc_homicides_2019_2024.csv")
cdc = cdc[cdc["name"] != "District of Columbia"].rename(columns={"name":"state","period":"year","count_sup":"homicide_count","rate":"homicide_rate"}).copy()
cdc["year"] = cdc["year"].astype(int)
ccw = pd.read_csv(p/"processed/rand_ccw_2019_2024.csv")
bg = pd.read_csv(p/"processed/rand_background_2019_2024.csv")
bls = pd.read_csv(p/"processed/bls_unemployment_2019_2024.csv")
bea = pd.read_csv(p/"processed/bea_income_2019_2024.csv")
print("Tabelas carregadas:",[(len(x),x["state"].nunique()) for x in [cdc,ccw,bg,bls,bea]])

## Bloco 2: Juntar e criar o alvo

Cada linha junta informações de um estado em um ano. `shift(-1)` procura a taxa do **ano seguinte** do mesmo estado; essa será a resposta a prever.

In [ ]:
colunas = ["state","year","homicide_count","homicide_rate"]
conjunto = cdc[colunas].merge(ccw,on=["state","year"],validate="one_to_one").merge(bg,on=["state","year"],validate="one_to_one").merge(bls,on=["state","year"],validate="one_to_one").merge(bea,on=["state","year"],validate="one_to_one")
conjunto = conjunto.sort_values(["state","year"]).reset_index(drop=True)
conjunto["ano_seguinte"] = conjunto.groupby("state")["year"].shift(-1)
assert (conjunto.loc[conjunto["ano_seguinte"].notna(),"ano_seguinte"] == conjunto.loc[conjunto["ano_seguinte"].notna(),"year"]+1).all()
conjunto["homicide_rate_next_year"] = conjunto.groupby("state")["homicide_rate"].shift(-1)
final = conjunto.dropna(subset=["homicide_rate_next_year"]).drop(columns="ano_seguinte")
print("Base para ML:",final.shape)
display(final.head())

## Bloco 3: Validar e salvar V3

Os 50 registros de 2024 ficam sem resposta de 2025 neste recorte e não entram no treinamento. Conferimos 250 linhas, 13 colunas e nenhum valor ausente.

In [ ]:
assert final.shape == (250,13)
assert final["state"].nunique() == 50
assert not final.duplicated(["state","year"]).any()
assert not final.isna().any().any()
destino = p/"processed/us_gun_policy_supervised_v3.csv"
final.to_csv(destino,index=False)
print("V3 salva:",destino.relative_to(ROOT), "|",final.shape)